<!-- notebook-header -->
# Estatistica Bayesiana para ML

**Modulo:** 01 - Estatistica  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Priors, likelihood, posterior, modelos conjugados, Naive Bayes, MAP e regularizacao.


# Estatistica Bayesiana para ML

## Indice

1. [Frequentista vs Bayesiano: Perspectivas Filosoficas](#1)
2. [Teorema de Bayes Revisitado](#2)
3. [Conjugate Priors: Beta-Binomial](#3)
4. [Atualizacao Bayesiana Sequencial](#4)
5. [Naive Bayes: Classificador Probabilistico](#5)
6. [Naive Bayes para Texto](#6)
7. [Classificador de Spam Completo](#7)
8. [Credible Intervals vs Confidence Intervals](#8)
9. [MAP e Conexao com Regularizacao](#9)
10. [Exercicios Praticos](#10)
11. [Erros Comuns e Armadilhas](#11)
12. [Resumo e Conexoes](#12)

## Pre-requisitos e Fio Narrativo

| Conceito | Notebook | Importancia |
|----------|----------|-------------|
| Testes de Hipotese (p-value, H0/H1) | 1_2 | Fundamental |
| Teorema de Bayes basico | 0_5 | Alta |
| Probabilidade Condicional P(A|B) | 0_5 | Alta |
| Distribuicoes (Normal, Beta) | 0_6 | Media |

**Tempo estimado:** 10-12 horas

**Fio narrativo:** No notebook 1_2, voce aprendeu a testar hipoteses no framework FREQUENTISTA: definir H0, calcular p-value, rejeitar ou nao. Agora voce vai conhecer uma ALTERNATIVA: o framework bayesiano. Em vez de perguntar "qual a probabilidade de ver esses dados se H0 for verdadeira?", o bayesiano pergunta "qual a probabilidade de minha hipotese ser verdadeira DADOS os dados que vi?" Essa inversao resolve muitos dos problemas discutidos no 1_2 (interpretacao de p-value, multiplos testes).

**Objetivos de aprendizado:**
- Compreender a filosofia bayesiana vs frequentista
- Aplicar o teorema de Bayes em problemas praticos
- Usar priors conjugados (Beta-Binomial) para atualizacao sequencial
- Implementar Naive Bayes para classificacao
- Distinguir credible intervals de confidence intervals
- Conectar MAP com regularizacao em ML

## Por que Estatistica Bayesiana em ML?

O pensamento bayesiano aparece em muitos algoritmos e praticas de ML:

- **Naive Bayes**: Classificador rapido e eficiente baseado diretamente no teorema de Bayes
- **Regularizacao**: L2 (Ridge) e equivalente a um prior Normal sobre os pesos; L1 (Lasso) e equivalente a um prior Laplace
- **Bayesian Neural Networks**: Quantificam incerteza nas predicoes (crucial para medicina, veiculos autonomos)
- **A/B testing bayesiano**: Permite "peeking" sem inflacao de falsos positivos (problema do 1_2)
- **Gaussian Processes**: Regressao bayesiana nao-parametrica com incerteza calibrada
- **Hyperparameter tuning**: Bayesian Optimization (ex: Optuna) usa posteriors para escolher proximos pontos
- **Transfer learning**: O modelo pre-treinado e essencialmente um "prior informativo"
- **Online learning**: Atualizacao sequencial do posterior conforme novos dados chegam

## 1. Frequentista vs Bayesiano: Perspectivas Filosoficas

### Analogia: Duas Maneiras de Pensar sobre Incerteza

Voce flipa uma moeda 5 vezes e obtem 4 caras (80%).

**Frequentista**: "Nao sei qual e a probabilidade verdadeira. Mas se repetisse esse experimento infinitas vezes e calculasse a media, chegaria a verdade. A probabilidade de cara e um valor FIXO que eu nao sei."

**Bayesiano**: "Comecei acreditando que a probabilidade era 50% (meu prior). Agora vi 4 caras em 5 flips - isso e evidencia. Vou ATUALIZAR minha crenca: agora acho que e ~70% (meu posterior). Se visse mais 100 flips, atualizaria novamente."

A diferenca crucial:
- **Frequentista**: Parametro e FIXO (desconhecido), dados sao aleatorios
- **Bayesiano**: Parametro e ALEATORIO (tem distribuicao de probabilidade), dados sao fixos

### Definicao Formal

| Aspecto | Frequentista | Bayesiano |
|---------|-------------|-----------|
| Parametro | Fixo, desconhecido | Aleatorio, tem distribuicao |
| Probabilidade | Frequencia em repeticoes | Grau de crenca |
| Inferencia | P(dados parametro) | P(parametro dados) |
| Intervalo | Confianca (propriedade do procedimento) | Credibilidade (probabilidade direta) |
| Vantagem | Objetividade | Interpretabilidade, sequencial |

### Por que em ML?

Muitos debates em ML sao frequentista vs bayesiano disfarçados. MLE (Maximum Likelihood) e frequentista; MAP (Maximum A Posteriori) e bayesiano. Regularizacao L2 e secretamente bayesiana (prior Normal). Entender ambas perspectivas permite escolher a melhor ferramenta para cada situacao.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import binom, norm, beta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')

print('=== FREQUENTISTA vs BAYESIANO ===')
print('\nFREQUENTISTA:')
print('  - Parametros sao fixos (desconhecidos, mas constantes)')
print('  - Dados sao aleatorios')
print('  - Probabilidade = frequencia em infinitas repeticoes')
print('  - Intervalo de confianca: se repetissemos 100 vezes, 95 conteriam o parametro')

print('\nBAYESIANO:')
print('  - Parametros tem distribuicao de probabilidade (incerteza)')
print('  - Dados sao fixos (observados)')
print('  - Probabilidade = grau de crenca')
print('  - Intervalo de credibilidade: 95% de probabilidade do parametro estar nele')


### O que observar

- O output mostra as duas perspectivas lado a lado para o mesmo problema
- Para o frequentista, a probabilidade de cara e um numero fixo (0.5 ou nao 0.5) - nao faz sentido dizer "probabilidade do parametro"
- Para o bayesiano, comeca com uma crenca (prior) e atualiza com dados
- A atualizacao sequencial e uma vantagem enorme: nao precisa refazer todo o calculo a cada novo dado

### O que concluir

- Nenhuma abordagem e "correta" - sao frameworks diferentes para a mesma realidade
- Frequentista e mais simples para testes padrao com amostras grandes
- Bayesiano e mais natural para decisoes sequenciais, incerteza sobre parametros, e incorporacao de conhecimento previo
- Em ML pratico, ambas coexistem: regularizacao e bayesiana, cross-validation e frequentista

### Conexao com outros notebooks

- **1_2 (Inferencial)**: Framework frequentista completo (H0, p-value, IC) - agora vemos a alternativa
- **0_5 (Probabilidade)**: Probabilidade condicional P(A|B) e a base do teorema de Bayes
- **0_6 (Distribuicoes)**: Distribuicoes Beta, Normal usadas como priors e posteriors

## 2. Teorema de Bayes Revisitado

### Analogia: Invertendo a Direcao da Probabilidade

O Teorema de Bayes "inverte" probabilidades condicionais. Voce sabe P(teste+ | doenca). Quer P(doenca | teste+). Bayes faz essa inversao.

**Exemplo do Teste Medico**: Um teste com 95% de acuracia (sensibilidade e especificidade) da positivo. Intuitivamente, parece que ha 95% de chance de ter a doenca. MAS: se a doenca e rara (1 em 1000), mesmo com teste 95% acurado, a probabilidade posterior e apenas ~2%! O prior (prevalencia) domina.

### Definicao Formal

P(theta | D) = P(D | theta) * P(theta) / P(D)

Posterior = Likelihood * Prior / Evidencia

Componentes:
- **P(theta | D) = Posterior**: distribuicao do parametro APOS observar dados
- **P(D | theta) = Likelihood**: probabilidade dos dados DADO o parametro
- **P(theta) = Prior**: crenca ANTES dos dados
- **P(D) = Evidencia**: constante de normalizacao (geralmente calculada como soma/integral)

### Por que em ML?

O teorema de Bayes e literalmente a equacao que fundamenta Naive Bayes (um dos classificadores mais usados em NLP/spam). Tambem e a base conceitual de toda regularizacao: o prior sobre parametros penaliza modelos complexos, prevenindo overfitting.

In [ ]:
print('=== TEOREMA DE BAYES ===' )
print(f'\nP(θ|D) = P(D|θ) × P(θ) / P(D)')
print(f'\nPosterior = Likelihood × Prior / Evidência')
print(f'\nComponentes:')
print(f'  P(θ|D) = Posterior: distribuição do parâmetro após dados')
print(f'  P(D|θ) = Likelihood: probabilidade dos dados dado parâmetro')
print(f'  P(θ) = Prior: crença inicial sobre o parâmetro')
print(f'  P(D) = Evidência: probabilidade marginal dos dados')

# Exemplo: Diagnóstico de doença
print(f'\n--- EXEMPLO: DIAGNÓSTICO MÉDICO ---')
print(f'Doença rara com prevalência 0.1%')
print(f'Teste com acurácia 95% (sensibilidade e especificidade)')
print(f'Pergunta: Se teste positivo, qual prob de ter doença?')

prior_disease = 0.001  # 0.1%
prior_no_disease = 1 - prior_disease

sensitivity = 0.95  # P(+|doença)
specificity = 0.95  # P(-|sem doença)

likelihood_positive_given_disease = sensitivity
likelihood_positive_given_no_disease = 1 - specificity

evidence = (likelihood_positive_given_disease * prior_disease + 
            likelihood_positive_given_no_disease * prior_no_disease)

posterior_disease = (likelihood_positive_given_disease * prior_disease) / evidence
posterior_no_disease = (likelihood_positive_given_no_disease * prior_no_disease) / evidence

print(f'\nCálculo:')
print(f'Prior (doença): {prior_disease:.4f} (0.1%)')
print(f'Likelihood (+|doença): {likelihood_positive_given_disease:.2f}')
print(f'Likelihood (+|sem doença): {likelihood_positive_given_no_disease:.2f}')
print(f'\nEvidência (marginal): {evidence:.4f}')
print(f'Posterior (doença|+): {posterior_disease:.4f} ({posterior_disease*100:.2f}%)')
print(f'\nInterpretação: Mesmo com teste 95% acurado, resultado + apenas')
print(f'aumenta probabilidade de doença para ~2%, não 95%! (devido ao prior baixo)')


### O que observar

- Mesmo com teste 95% acurado, P(doenca | +) e apenas ~2% quando prevalencia e 0.1%
- O prior (prevalencia = 0.001) domina completamente o resultado
- O denominador (evidencia) e ~0.05, que inclui tanto os verdadeiros positivos quanto os falsos positivos
- A maioria dos positivos sao FALSOS positivos porque a doenca e muito rara

### O que concluir

- O teorema de Bayes mostra que a probabilidade posterior depende TANTO da evidencia (likelihood) QUANTO da crenca previa (prior)
- Com prior forte (doenca rara), precisa-se de evidencia muito forte para mudar a conclusao
- Isso explica por que testes de screening em massa geram muitos falsos positivos
- A mesma logica se aplica em ML: um classificador pode ter alta acuracia mas baixa precisao se a classe positiva for rara (desbalanceamento)

### Conexao com outros notebooks

- **0_5 (Probabilidade)**: Teorema de Bayes introduzido la; aqui vemos aplicacao completa com calculo numerico
- **1_2 (Inferencial)**: O p-value frequentista P(dados | H0) e DIFERENTE de P(H0 | dados) bayesiano - essa confusao e o "erro 1" do 1_2
- **1_1 (Descritiva)**: Prevalencia (prior) e uma estatistica descritiva da populacao

## 3. Conjugate Priors: Beta-Binomial

### Analogia: Priors que "Combinam" com os Dados

Quando o prior e de uma familia matematica que, combinada com a likelihood, produz um posterior da MESMA familia, temos um "par conjugado". Isso simplifica enormemente os calculos.

O par mais famoso e **Beta-Binomial**: se o prior sobre uma proporcao e Beta(a, b) e os dados sao binomiais (s sucessos em n tentativas), o posterior e Beta(a + s, b + n - s). Simples!

### Definicao Formal

- Prior: theta ~ Beta(alpha, beta)
- Dados: k sucessos em n tentativas (Binomial)
- Posterior: theta | dados ~ Beta(alpha + k, beta + n - k)

A media posterior e (alpha + k) / (alpha + beta + n), que e uma media ponderada entre o prior e o MLE:
- MLE (frequentista): k/n
- Prior mean: alpha/(alpha+beta)
- Posterior mean: combinacao ponderada dos dois

### Por que em ML?

Priors conjugados aparecem em Bayesian Optimization (prior sobre funcao objetivo), em modelos de topicos (LDA usa Dirichlet-Multinomial, que e a versao multidimensional de Beta-Binomial), e em A/B testing bayesiano (posterior Beta para taxas de conversao).

In [ ]:
print('=== CONJUGATE PRIORS ===' )
print(f'\nPar conjugado: Prior e Likelihood têm a mesma forma')
print(f'→ Posterior também tem essa forma!')
print(f'→ Cálculo analítico é simples')

# Beta-Binomial (para taxa/proporção)
print(f'\n1. BETA-BINOMIAL: Estimar Taxa de Conversão')
print(f'\nCenário: Um site tem CTR desconhecida')
print(f'Prior: Beta(α=2, β=2) [crença fraca, centrada em 0.5]')
print(f'Dados: 8 cliques em 20 impressões')

alpha_prior = 2
beta_prior = 2
clicks = 8
impressns = 20

alpha_posterior = alpha_prior + clicks
beta_posterior = beta_prior + (impressns - clicks)

print(f'\nPrior: Beta({alpha_prior}, {beta_prior})')
print(f'Likelihood: Binomial({impressns}, cliques={clicks})')
print(f'Posterior: Beta({alpha_posterior}, {beta_posterior})')

mean_prior = alpha_prior / (alpha_prior + beta_prior)
mean_posterior = alpha_posterior / (alpha_posterior + beta_posterior)
mean_mle = clicks / impressns

print(f'\nMédias:')
print(f'  Prior: {mean_prior:.3f}')
print(f'  MLE (frequentista): {mean_mle:.3f}')
print(f'  Posterior (bayesiano): {mean_posterior:.3f}')

# Intervalo de credibilidade
ic_lower = beta.ppf(0.025, alpha_posterior, beta_posterior)
ic_upper = beta.ppf(0.975, alpha_posterior, beta_posterior)
print(f'\nCredible Interval 95%: [{ic_lower:.3f}, {ic_upper:.3f}]')

# Visualização
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

x = np.linspace(0, 1, 100)

axes[0].plot(x, beta.pdf(x, alpha_prior, beta_prior), linewidth=2, label=f'Prior Beta({alpha_prior}, {beta_prior})')
axes[0].fill_between(x, 0, beta.pdf(x, alpha_prior, beta_prior), alpha=0.3)
axes[0].set_xlabel('Taxa de Conversão')
axes[0].set_ylabel('Densidade')
axes[0].set_title('Prior Conjugado (Beta)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(x, beta.pdf(x, alpha_posterior, beta_posterior), linewidth=2, color='orange', label=f'Posterior Beta({alpha_posterior}, {beta_posterior})')
axes[1].fill_between(x, 0, beta.pdf(x, alpha_posterior, beta_posterior), alpha=0.3, color='orange')
axes[1].axvline(ic_lower, color='red', linestyle='--', alpha=0.5, label=f'95% CI')
axes[1].axvline(ic_upper, color='red', linestyle='--', alpha=0.5)
axes[1].axvline(mean_posterior, color='green', linestyle='-', linewidth=2, alpha=0.7, label='Média posterior')
axes[1].set_xlabel('Taxa de Conversão')
axes[1].set_ylabel('Densidade')
axes[1].set_title(f'Posterior após Dados (8/20)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(x, beta.pdf(x, alpha_prior, beta_prior), linewidth=2, label='Prior', alpha=0.5)
axes[2].plot(x, beta.pdf(x, alpha_posterior, beta_posterior), linewidth=2, label='Posterior', alpha=0.8)
axes[2].axvline(mean_mle, color='red', linestyle='--', linewidth=2, label=f'MLE ({mean_mle:.3f})')
axes[2].set_xlabel('Taxa de Conversão')
axes[2].set_ylabel('Densidade')
axes[2].set_title('Prior vs Posterior vs MLE')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### O que observar

- O prior Beta(2,2) e uma curva suave centrada em 0.5 (crenca fraca de que a moeda e justa)
- Apos 8/20 cliques, o posterior Beta(10, 14) se desloca para a esquerda do MLE (0.4)
- A media posterior (0.417) fica ENTRE o prior (0.5) e o MLE (0.4) - e uma "media ponderada"
- O IC de credibilidade 95% e interpretavel: "95% de probabilidade de que a taxa esta nesse intervalo"

### O que concluir

- Com poucos dados (n=20), o prior tem influencia significativa - puxa a estimativa na direcao da crenca previa
- Com muitos dados, o posterior converge para o MLE (os dados dominam o prior)
- A "forca" do prior e controlada por alpha + beta: quanto maior, mais dados sao necessarios para mudar a crenca
- Beta(1,1) = Uniforme e o prior "nao-informativo" mais comum para proporcoes

### Conexao com outros notebooks

- **0_6 (Distribuicoes)**: A distribuicao Beta estudada la agora e usada como prior e posterior
- **1_2 (Inferencial)**: IC frequentista vs IC de credibilidade bayesiano para a mesma metrica
- **0_8 (Otimizacao)**: Bayesian Optimization usa priors conjugados (ou Gaussian Processes) para otimizar hiperparametros

## 4. Atualizacao Bayesiana Sequencial

### Analogia: Aprendendo Passo a Passo

Uma das grandes vantagens bayesianas e a atualizacao SEQUENCIAL: o posterior de hoje vira o prior de amanha. A cada novo dado, a crenca se refina.

Imagine estimar a probabilidade de uma moeda:
- Dia 1: Comeca com Beta(1,1) [nenhum conhecimento]. Observa 7 caras em 10 flips -> Beta(8,4)
- Dia 2: Usa Beta(8,4) como prior. Observa 6 caras em 10 flips -> Beta(14,8)
- Dia 3: Usa Beta(14,8) como prior. E assim por diante...

Nao precisa reprocessar TAREFA DO ALUNOS os dados cada vez! O posterior "resume" toda a informacao passada.

### Definicao Formal

Dado posterior atual Beta(alpha_t, beta_t) e novos dados (k sucessos em n tentativas):
alpha_{t+1} = alpha_t + k
beta_{t+1} = beta_t + (n - k)

### Por que em ML?

Atualizacao sequencial e a base de ONLINE LEARNING: atualizar o modelo conforme novos dados chegam, sem retreinar do zero. Tambem e essencial em sistemas de recomendacao (atualizar preferencias do usuario), A/B testing adaptivo (parar o teste quando ha evidencia suficiente), e deteccao de anomalias em streaming.

In [ ]:
print('=== ATUALIZAÇÃO SEQUENCIAL ===' )

# Simulação: Estimando moeda enviesada
np.random.seed(42)

prob_true = 0.7  # Verdadeira probabilidade de cara
sequence = np.random.binomial(1, prob_true, 100)

alpha = 1  # Prior não-informativo: Beta(1, 1) = Uniform
beta_param = 1

alphas = [alpha]
betas = [beta_param]

for i, flip in enumerate(sequence):
    if flip == 1:
        alpha += 1
    else:
        beta_param += 1
    
    if i % 10 == 9:  # Salve a cada 10 flips
        alphas.append(alpha)
        betas.append(beta_param)

alphas.append(alpha)
betas.append(beta_param)

print(f'Moeda verdadeira: P(cara) = {prob_true}')
print(f'\nEvoluução da crença:')
for i, (a, b) in enumerate([(alphas[0], betas[0])] + list(zip(alphas[1::11], betas[1::11]))):
    n_obs = a + b - 2 if i == 0 else (a-1) + (b-1)
    mean = a / (a + b)
    print(f'  Após {n_obs:2d} observações: Beta({a:2d}, {b:2d}), Média = {mean:.3f}')

# Visualização
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

x = np.linspace(0, 1, 100)

# Prior
axes[0, 0].plot(x, beta.pdf(x, 1, 1), linewidth=2, label='Prévio (Uniforme)')
axes[0, 0].fill_between(x, 0, beta.pdf(x, 1, 1), alpha=0.3)
axes[0, 0].axvline(0.5, color='red', linestyle='--', alpha=0.5, label='E[θ]')
axes[0, 0].set_xlabel('Probabilidade de Cara')
axes[0, 0].set_ylabel('Densidade')
axes[0, 0].set_title('Prior (antes de observações)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Após 10
axes[0, 1].plot(x, beta.pdf(x, alphas[1], betas[1]), linewidth=2, color='orange')
axes[0, 1].fill_between(x, 0, beta.pdf(x, alphas[1], betas[1]), alpha=0.3, color='orange')
axes[0, 1].axvline(alphas[1]/(alphas[1] + betas[1]), color='red', linestyle='--', alpha=0.5)
axes[0, 1].axvline(prob_true, color='green', linestyle='--', linewidth=2, alpha=0.7, label='Verdadeira')
axes[0, 1].set_xlabel('Probabilidade de Cara')
axes[0, 1].set_ylabel('Densidade')
axes[0, 1].set_title(f'Posterior após 10 flips')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xlim(0, 1)

# Após 50
axes[1, 0].plot(x, beta.pdf(x, alphas[5], betas[5]), linewidth=2, color='lightcoral')
axes[1, 0].fill_between(x, 0, beta.pdf(x, alphas[5], betas[5]), alpha=0.3, color='lightcoral')
axes[1, 0].axvline(alphas[5]/(alphas[5] + betas[5]), color='red', linestyle='--', alpha=0.5)
axes[1, 0].axvline(prob_true, color='green', linestyle='--', linewidth=2, alpha=0.7, label='Verdadeira')
axes[1, 0].set_xlabel('Probabilidade de Cara')
axes[1, 0].set_ylabel('Densidade')
axes[1, 0].set_title(f'Posterior após 50 flips')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xlim(0, 1)

# Evolução das médias
means = [a / (a + b) for a, b in zip(alphas, betas)]
axes[1, 1].plot(np.arange(len(means)) * 10, means, 'o-', linewidth=2, markersize=6, label='Média posterior')
axes[1, 1].axhline(prob_true, color='green', linestyle='--', linewidth=2, label='Verdadeira prob.')
axes[1, 1].set_xlabel('Número de Observações')
axes[1, 1].set_ylabel('Probabilidade de Cara')
axes[1, 1].set_title('Convergência da Crença')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

### O que observar

- O prior Uniforme (Dia 0) e completamente "aberto" - qualquer valor de 0 a 1 e igualmente provavel
- Apos 10 flips, o posterior ja comeca a se concentrar em torno de 0.7
- Apos 50 flips, o posterior e uma curva estreita e precisa
- A media posterior CONVERGE para o valor verdadeiro (0.7) conforme n cresce

### O que concluir

- Com poucos dados, a distribuicao posterior e larga (muita incerteza)
- Com muitos dados, a posterior se estreita e concentra no valor verdadeiro (independente do prior)
- O prior importa MUITO com poucos dados e POUCO com muitos dados
- A atualizacao sequencial da exatamente o mesmo resultado que processar todos os dados de uma vez (propriedade fundamental)
- A velocidade de convergencia depende da forca do prior e da variabilidade dos dados

### Conexao com outros notebooks

- **0_7 (TCL)**: A convergencia do posterior para Normal com n grande e analoga ao Teorema Central do Limite
- **0_8 (Otimizacao)**: Online learning (SGD) tambem atualiza parametros sequencialmente, mas sem framework probabilistico
- **1_2 (Inferencial)**: Atualizacao sequencial resolve o problema de "peeking" em A/B testing

## 5. Naive Bayes: Classificador Probabilistico

### Analogia: "Quais Features Apontam para Qual Classe?"

Naive Bayes aplica o teorema de Bayes para classificacao:
P(classe | features) proporcional a P(features | classe) * P(classe)

O "naive" (ingenuo) vem da suposicao de que as features sao INDEPENDENTES dada a classe:
P(f1, f2, f3 | classe) = P(f1 | classe) * P(f2 | classe) * P(f3 | classe)

Essa suposicao e quase nunca verdadeira na pratica, mas o classificador funciona surpreendentemente bem mesmo assim!

### Definicao Formal

- **GaussianNB**: Assume features continuas com distribuicao Normal por classe
- **Multinomial
NB**: Para contagens (ex: frequencia de palavras em texto)
- **BernoulliNB**: Para features binarias (presenca/ausencia)

### Por que em ML?

Naive Bayes e o classificador padrao para:
- Filtros de spam (Multinomial
NB sobre contagem de palavras)
- Classificacao de sentimento em texto
- Deteccao de fraude em transacoes
- Diagnostico medico com multiplos sintomas

Suas vantagens: treinamento rapido (O(n*d)), funciona bem com poucos dados, fornece probabilidades calibradas, e escala linearmente.

In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('=== NAIVE BAYES ===' )
print(f'\nIdeia: P(classe|features) ∝ P(features|classe) × P(classe)')
print(f'"Naive": Assume independência entre features dado a classe')
print(f'P(f1, f2, f3|classe) = P(f1|classe) × P(f2|classe) × P(f3|classe)')
print(f'(Raramente verdade, mas funciona bem na prática!)')

# Dataset
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f'\n--- DATASET IRIS ---')
print(f'Amostras: {X.shape[0]}, Features: {X.shape[1]}, Classes: {len(np.unique(y))}')
print(f'Treino: {X_train.shape[0]}, Teste: {X_test.shape[0]}')

# Gaussian Naive Bayes
model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'\nGaussian Naive Bayes:')
print(f'Acurácia: {accuracy:.3f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# Priors
print(f'\nPriors (probabilidades de classe no treino):')
for i, class_name in enumerate(iris.target_names):
    prior = (y_train == i).mean()
    print(f'  {class_name}: {prior:.3f}')

# Likelihoods (médias e variâncias por feature e classe)
print(f'\nExemplo de Likelihood: Sepal Length por Classe')
for i, class_name in enumerate(iris.target_names):
    feature_data = X_train[y_train == i, 0]  # Sepal length
    mean = feature_data.mean()
    var = feature_data.var()
    print(f'  {class_name}: μ={mean:.2f}, σ²={var:.2f}')


### O que observar

- GaussianNB treina instantaneamente (so calcula medias e variancias por classe/feature)
- Os priors sao simplesmente as proporcoes das classes no treino
- As likelihoods sao parametrizadas por media e variancia de cada feature em cada classe
- A acuracia e alta (~97%) mesmo com a suposicao "ingenua" de independencia

### O que concluir

- Naive Bayes funciona bem quando features nao sao fortemente correlacionadas entre si
- E um baseline excelente: rapido, interpretavel, e surpreendentemente competitivo
- A suposicao de independencia e violada em quase todos os datasets reais, mas as PROBABILIDADES podem ser mal calibradas enquanto as CLASSIFICACOES continuam corretas
- Para calibrar probabilidades, use Calibrated
Classifier
CV do sklearn

### Conexao com outros notebooks

- **0_5 (Probabilidade)**: Independencia condicional P(A,B|C) = P(A|C)*P(B|C) e a suposicao "naive"
- **1_1 (Descritiva)**: Media e variancia por classe sao estatisticas descritivas condicionais
- **1_4 (Regressao)**: Regressao logistica e uma alternativa discriminativa ao Naive Bayes generativo

## 6. Naive Bayes para Texto

### Analogia: Contar Palavras para Classificar

Em classificacao de texto, cada documento e representado por contagens de palavras (bag of words). Multinomial
NB calcula: para cada palavra, qual a probabilidade de ela aparecer em documentos de cada classe?

Se "viagra" aparece 100x em spam e 1x em emails legitimos, e forte evidencia para spam. Naive Bayes multiplica essas evidencias palavra por palavra.

### Definicao Formal

P(classe | documento) proporcional a P(classe) * PRODUTO_i P(palavra_i | classe)

Na pratica, usa-se log-probabilidades para evitar underflow numerico:
log P(classe | doc) = log P(classe) + SOMA_i log P(palavra_i | classe)

Laplace smoothing (alpha=1) evita probabilidade zero para palavras novas.

### Por que em ML?

Multinomial
NB e o classificador mais usado em NLP classica (antes de deep learning). Ainda e competitivo para: classificacao de emails, analise de sentimento em reviews, categorizacao de documentos, e deteccao de toxicidade em comentarios. E ordens de magnitude mais rapido que transformers.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

print('=== NAIVE BAYES PARA TEXTO ===' )

# Dados de exemplo
docs = [
    'spam spam spam viagra',
    'spam viagra cheap',
    'legitimate email from boss',
    'meeting at 3pm tomorrow',
    'urgent: buy cheap drugs',
    'conference schedule attached'
]

labels = [1, 1, 0, 0, 1, 0]  # 1=spam, 0=legit

print(f'\nDataset: {len(docs)} emails')
for i, (doc, label) in enumerate(zip(docs, labels)):
    print(f'  {i+1}. ["{doc}"] → {"SPAM" if label else "LEGIT"}')

# Vetorização (contagem de palavras)
vectorizer = CountVectorizer()
X_vec = vectorizer.fit_transform(docs)

print(f'\nVocabulário: {vectorizer.get_feature_names_out()}')
print(f'\nMatriz de contagem (primeiros 3 emails):')
print(X_vec[:3].toarray())

# Multinomial Naive Bayes
model = MultinomialNB()
model.fit(X_vec, labels)

y_pred = model.predict(X_vec)
accuracy = accuracy_score(labels, y_pred)

print(f'\nMultinomial Naive Bayes:')
print(f'Acurácia (treino): {accuracy:.3f}')

# Predição em novo texto
new_text = ['spam viagra cheap']
X_new = vectorizer.transform(new_text)
pred = model.predict(X_new)[0]
proba = model.predict_proba(X_new)[0]

print(f'\nPredicao: "{new_text[0]}"')
print(f'  P(LEGIT) = {proba[0]:.4f}')
print(f'  P(SPAM) = {proba[1]:.4f}')
print(f'  Classe: {"SPAM" if pred else "LEGIT"}')

# Log-probabilidades (features mais indicativas)
log_probs = model.feature_log_prob_
feature_names = vectorizer.get_feature_names_out()

print(f'\nPalavras mais indicativas de SPAM:')
spam_log_probs = log_probs[1] - log_probs[0]  # Diferença log-prob
top_spam_idx = np.argsort(spam_log_probs)[-3:]
for idx in reversed(top_spam_idx):
    print(f'  {feature_names[idx]}: {spam_log_probs[idx]:.2f}')


### O que observar

- CountVectorizer transforma texto em matriz de contagem de palavras (bag of words)
- Palavras como "spam", "viagra", "cheap" tem alta probabilidade condicional na classe spam
- A diferenca de log-probabilidades mostra QUAIS palavras sao mais indicativas de cada classe
- predict_proba retorna probabilidades (nao so a classe), util para definir thresholds

### O que concluir

- Bag of words ignora ordem das palavras (perde contexto), mas funciona bem para classificacao simples
- Laplace smoothing (alpha=1) e essencial para evitar probabilidade zero quando uma palavra nao aparece em uma classe
- Em datasets pequenos, Multinomial
NB frequentemente supera modelos mais complexos (menos overfitting)
- As log-probabilidades por feature dao interpretabilidade: voce sabe QUAIS palavras levam a classificacao

### Conexao com outros notebooks

- **0_3 (Algebra Linear)**: A matriz de contagem (term-document matrix) e um caso de representacao vetorial de dados
- **1_1 (Descritiva)**: Frequencias de palavras sao estatisticas descritivas do texto
- **0_8 (Otimizacao)**: TF-IDF e uma transformacao que pondera frequencias; pode ser vista como normalizacao

## 7. Classificador de Spam Completo

### Analogia: Pipeline de ML com Naive Bayes

Ate aqui vimos as pecas separadas. Agora montamos um pipeline completo:

1. Preprocessamento de texto (lowercase, stop words)
2. Vetorizacao (CountVectorizer)
3. Treinamento (Multinomial
NB com Laplace smoothing)
4. Validacao (cross-validation)
5. Predicao com probabilidades

### Por que em ML?

Este e exatamente o pipeline usado em producao para filtros de spam, classificadores de suporte ao cliente, e triagem automatica de tickets. A simplicidade do Naive Bayes permite processar milhoes de documentos por segundo.

In [ ]:
# Dados sintéticos mais realistas
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import cross_val_score

print('=== CLASSIFICADOR DE SPAM ===' )

# Dataset simplificado
training_docs = [
    'Click here to get rich quick',
    'Buy cheap medications now',
    'Limited time offer: amazing deals',
    'Meeting notes from yesterday',
    'Project deadline is Friday',
    'Quarterly earnings report attached',
    'URGENT: Send money NOW!!!',
    'Best prices on brand name drugs',
    'Team lunch tomorrow at noon',
    'New product launch announcement'
]

labels = [1, 1, 1, 0, 0, 0, 1, 1, 0, 0]  # 1=spam, 0=ham

vectorizer = CountVectorizer(lowercase=True, stop_words='english')
X = vectorizer.fit_transform(training_docs)

model = MultinomialNB(alpha=1.0)  # Laplace smoothing
model.fit(X, labels)

# Cross-validation
scores = cross_val_score(model, X, labels, cv=3)
print(f'Cross-validation scores: {scores}')
print(f'Média: {scores.mean():.3f}, Desvio padrão: {scores.std():.3f}')

# Testes de spam
test_emails = [
    'Join our exclusive club today',
    'Meeting postponed to next week',
    'Free money just click here',
    'Budget review scheduled'
]

print(f'\nTestando novos emails:')
X_test = vectorizer.transform(test_emails)
y_test_pred = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)

for email, pred, proba in zip(test_emails, y_test_pred, y_test_proba):
    label = 'SPAM' if pred == 1 else 'HAM'
    confidence = max(proba)
    print(f'\n  Email: "{email}"')
    print(f'  Previsão: {label} (confiança: {confidence:.2%})')


### O que observar

- Cross-validation com k=3 mostra variancia nos scores (dataset pequeno)
- Emails com palavras tipicas de spam ("free", "click", "money") sao classificados corretamente
- A confianca (probabilidade) varia: alguns emails sao classificados com mais certeza que outros
- stop_words='english' remove palavras comuns que nao ajudam na classificacao

### O que concluir

- Pipeline completo inclui preprocessamento, vetorizacao, treinamento E validacao
- Em datasets pequenos, cross-validation pode ter alta variancia; nao confie em um unico fold
- Laplace smoothing (alpha=1.0) e o default e geralmente funciona bem; ajustar alpha e uma forma de regularizacao
- Em producao, adicionaria: TF-IDF, n-grams, e feature engineering especifica do dominio

### Conexao com outros notebooks

- **1_2 (Inferencial)**: Cross-validation aqui e a mesma tecnica frequentista para estimar performance
- **1_5 (Design de Experimentos)**: Train/test split e uma forma basica de design experimental
- **0_8 (Otimizacao)**: Alpha do Laplace smoothing e um hiperparametro que poderia ser otimizado

## 8. Credible Intervals vs Confidence Intervals

### Analogia: Duas Interpretacoes do "Intervalo"

Esta e uma das distincoes mais importantes (e confusas) em estatistica:

**Intervalo de Confianca (Frequentista)**: "Se eu repetisse o experimento 100 vezes, em ~95 vezes o verdadeiro parametro estaria no intervalo." O parametro e FIXO; o intervalo e que varia.

**Intervalo de Credibilidade (Bayesiano)**: "Ha 95% de probabilidade de que o parametro esta neste intervalo." O intervalo e fixo; o parametro e que tem distribuicao.

A interpretacao bayesiana e muito mais intuitiva! E geralmente o que as pessoas PENSAM que o IC frequentista significa (mas nao e).

### Definicao Formal

- IC frequentista: construido a partir da distribuicao amostral; P(parametro em IC) nao e definido
- IC de credibilidade: derivado diretamente da distribuicao posterior; P(parametro em IC) = 0.95

### Por que em ML?

Quando reporta metricas de modelo, a interpretacao bayesiana e muito mais natural: "ha 95% de probabilidade de que a acuracia real esta entre 82% e 88%." O frequentista diria algo muito mais convoluto. Para comunicar incerteza a stakeholders nao-tecnicos, credible intervals sao muito mais claros.

In [ ]:
print('=== CREDIBLE vs CONFIDENCE INTERVALS ===' )

print(f'\nINTERVALO DE CONFIANÇA (Frequentista):')
print(f'  "Se repetíssemos o experimento 100 vezes, em 95 vezes"')
print(f'   o verdadeiro parâmetro estaria no intervalo."')
print(f'  → Parâmetro é fixo, intervalo varia')
print(f'  → Não podemos dizer P(parâmetro ∈ IC) = 0.95')
print(f'   (parâmetro não é aleatório para frequentista)')

print(f'\nCREDIBLE INTERVAL (Bayesiano):')
print(f'  "Há 95% de probabilidade de o parâmetro estar neste intervalo."')
print(f'  → Parâmetro é aleatório (tem distribuição posterior)')
print(f'  → Intervalo é fixo')
print(f'  → Podemos fazer afirmações diretas sobre o parâmetro')

# Simulação visual
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Frequentist CI: múltiplas amostras
num_samples = 20
mu_true = 100
sigma = 10
n = 30

axes[0].axhline(mu_true, color='red', linestyle='-', linewidth=2, label='Parâmetro verdadeiro')

for i in range(num_samples):
    sample = np.random.normal(mu_true, sigma, n)
    x_bar = sample.mean()
    se = sigma / np.sqrt(n)
    ci_lower = x_bar - 1.96 * se
    ci_upper = x_bar + 1.96 * se
    
    color = 'blue' if mu_true >= ci_lower and mu_true <= ci_upper else 'red'
    axes[0].plot([ci_lower, ci_upper], [i, i], 'o-', color=color, alpha=0.6)

axes[0].set_xlabel('Valor')
axes[0].set_ylabel('Amostra')
axes[0].set_title('Frequentista: 20 Amostras, 20 ICs\n(~95% contêm o verdadeiro parâmetro)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bayesian CI: posterior
alpha_post = 50 + 25  # prior + dados
beta_post = 50 + 75

x = np.linspace(0, 1, 100)
axes[1].plot(x, beta.pdf(x, alpha_post, beta_post), linewidth=2, color='green')
axes[1].fill_between(x, 0, beta.pdf(x, alpha_post, beta_post), alpha=0.3, color='green')

ic_lower = beta.ppf(0.025, alpha_post, beta_post)
ic_upper = beta.ppf(0.975, alpha_post, beta_post)
axes[1].axvline(ic_lower, color='orange', linestyle='--', linewidth=2, label='95% Credible Interval')
axes[1].axvline(ic_upper, color='orange', linestyle='--', linewidth=2)
axes[1].axvline(alpha_post/(alpha_post + beta_post), color='red', linestyle='-', linewidth=2, alpha=0.7, label='Posterior mean')

axes[1].set_xlabel('Parâmetro')
axes[1].set_ylabel('Densidade')
axes[1].set_title('Bayesiano: Posterior Distribution\n(95% de probabilidade no intervalo)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### O que observar

- No grafico frequentista, cada barra horizontal e um IC de uma amostra diferente; ~1 em 20 NAO contem o parametro verdadeiro (vermelho)
- No grafico bayesiano, ha UMA distribuicao posterior com UM intervalo de credibilidade
- A interpretacao visual e diferente: frequentista mostra variabilidade do PROCEDIMENTO; bayesiano mostra incerteza sobre o PARAMETRO
- Numericamente, os dois intervalos podem ser muito similares (especialmente com priors fracos e amostras grandes)

### O que concluir

- Com amostras grandes e priors fracos, IC frequentista e IC bayesiano convergem para valores similares
- A diferenca e FILOSOFICA e de INTERPRETACAO, nao necessariamente numerica
- Para decisoes praticas, a interpretacao bayesiana e geralmente mais util e menos confusa
- O IC frequentista e uma propriedade do METAREFA DO ALUNO; o IC bayesiano e uma propriedade da CRENCA sobre o parametro

### Conexao com outros notebooks

- **1_2 (Inferencial)**: IC frequentista foi calculado la; agora vemos a alternativa bayesiana
- **0_6 (Distribuicoes)**: O posterior e uma distribuicao completa (Beta, Normal), nao so um ponto
- **1_4 (Regressao)**: ICs para coeficientes de regressao podem ser frequentistas ou bayesianos

## 9. MAP (Maximum A Posteriori) e Conexao com Regularizacao

### Analogia: "Qual o Ponto Mais Provavel do Posterior?"

Ate agora trabalhamos com a distribuicao posterior COMPLETA. Mas as vezes queremos apenas um unico "melhor" valor do parametro. Temos duas opcoes:

- **MLE (Maximum Likelihood)**: Escolhe theta que maximiza P(dados | theta). Ignora o prior.
- **MAP (Maximum A Posteriori)**: Escolhe theta que maximiza P(theta | dados) = P(dados | theta) * P(theta). Incorpora o prior.

A conexao surpreendente: MAP com prior Normal sobre os pesos e IDENTICO a regressao com regularizacao L2 (Ridge)! MAP com prior Laplace e identico a L1 (Lasso)!

### Definicao Formal

- MLE: argmax_theta P(D | theta) = argmax_theta PROD P(x_i | theta)
- MAP: argmax_theta P(D | theta) * P(theta) = MLE + penalidade do prior
- Com prior Normal(0, sigma^2): MAP = MLE + lambda * ||theta||^2 (Ridge!)
- Com prior Laplace(0, b): MAP = MLE + lambda * ||theta||_1 (Lasso!)

### Por que em ML?

Essa conexao e profunda: TODA regularizacao pode ser vista como um prior bayesiano. L2 (Ridge) diz "acredito que os pesos sao pequenos" (prior Normal). L1 (Lasso) diz "acredito que muitos pesos sao zero" (prior Laplace). Dropout pode ser visto como inferencia bayesiana aproximada. Essa perspectiva unifica dois mundos aparentemente separados.

In [ ]:
print('=== MAP vs MLE vs REGULARIZACAO ===')

# Demonstracao: estimar proporcao de cara em moeda
# MLE = k/n (puro dados)
# MAP com Beta(a,b) = (k+a-1)/(n+a+b-2) (modo do posterior)

n_flips = 10
k_caras = 9  # 9 caras em 10 flips

# MLE
mle = k_caras / n_flips

# MAP com diferentes priors
priors = [
    ('Uniforme Beta(1,1)', 1, 1),
    ('Fraco Beta(2,2)', 2, 2),
    ('Forte Beta(10,10)', 10, 10),
    ('Muito forte Beta(50,50)', 50, 50),
]

print(f'Dados: {k_caras} caras em {n_flips} flips')
print(f'\nMLE (frequentista): {mle:.3f}')
print(f'\nMAP com diferentes priors:')
for name, a, b in priors:
    # Modo do posterior Beta(a+k, b+n-k)
    a_post = a + k_caras
    b_post = b + (n_flips - k_caras)
    map_est = (a_post - 1) / (a_post + b_post - 2) if (a_post > 1 and b_post > 1) else a_post / (a_post + b_post)
    mean_post = a_post / (a_post + b_post)
    print(f'  {name}: MAP = {map_est:.3f}, Media posterior = {mean_post:.3f}')

# Conexao com regularizacao
print(f'\n=== CONEXAO COM REGULARIZACAO ===')
print(f'\nRegressao Linear:')
print(f'  MLE: min ||y - Xw||^2')
print(f'  MAP (prior Normal): min ||y - Xw||^2 + lambda * ||w||^2  [= Ridge/L2]')
print(f'  MAP (prior Laplace): min ||y - Xw||^2 + lambda * ||w||_1  [= Lasso/L1]')
print(f'\nQuanto mais forte o prior (menor sigma^2), maior o lambda:')
print(f'  lambda = sigma_dados^2 / sigma_prior^2')
print(f'\nAssim:')
print(f'  Prior amplo (sigma grande) -> lambda pequeno -> pouca regularizacao')
print(f'  Prior estreito (sigma pequeno) -> lambda grande -> muita regularizacao')

# Visualizacao
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAP vs MLE com diferentes priors
x = np.linspace(0, 1, 200)
colors = ['blue', 'green', 'orange', 'red']

for (name, a, b), color in zip(priors, colors):
    a_post = a + k_caras
    b_post = b + (n_flips - k_caras)
    axes[0].plot(x, beta.pdf(x, a_post, b_post), linewidth=2, color=color, label=f'{name}')

axes[0].axvline(mle, color='black', linestyle='--', linewidth=2, label=f'MLE = {mle:.2f}')
axes[0].set_xlabel('Probabilidade de Cara')
axes[0].set_ylabel('Densidade')
axes[0].set_title(f'Posterior com Diferentes Priors ({k_caras}/{n_flips} caras)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Analogia visual: prior = regularizacao
lambdas = [0, 0.1, 1, 10]
w = np.linspace(-3, 3, 200)

for lam, color in zip(lambdas, colors):
    # Likelihood (dados) + Prior (regularizacao)
    likelihood = norm.pdf(w, loc=2, scale=0.5)  # dados sugerem w=2
    if lam == 0:
        posterior = likelihood
        label = f'MLE (lambda=0)'
    else:
        prior = norm.pdf(w, loc=0, scale=1/np.sqrt(lam))
        posterior = likelihood * prior
        posterior = posterior / (posterior.sum() * (w[1]-w[0]))
        label = f'MAP (lambda={lam})'
    axes[1].plot(w, posterior, linewidth=2, color=color, label=label)

axes[1].set_xlabel('Peso (w)')
axes[1].set_ylabel('Densidade')
axes[1].set_title('MAP = MLE + Regularizacao\n(Prior Normal = Ridge/L2)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### O que observar

- Com prior fraco Beta(1,1), MAP ≈ MLE (0.9) - os dados dominam
- Com prior forte Beta(50,50), MAP e puxado para 0.5 mesmo com 9/10 caras - o prior domina
- No grafico de regularizacao, lambda=0 (MLE) poe o peso em w=2 (puro dado); lambda=10 puxa para w=0 (puro prior)
- A transicao de lambda=0 para lambda grande e EXATAMENTE a transicao de MLE para MAP com prior cada vez mais forte

### O que concluir

- MAP e MLE diferem APENAS pelo prior; sem prior (ou prior uniforme), MAP = MLE
- Regularizacao L2 (Ridge) = prior Normal centrado em zero sobre os pesos
- Regularizacao L1 (Lasso) = prior Laplace centrado em zero (promove sparsity)
- O hiperparametro lambda de regularizacao e a "forca" do prior bayesiano
- Essa conexao unifica a visao frequentista (regularizacao = penalidade) com a bayesiana (regularizacao = prior)

### Conexao com outros notebooks

- **0_8 (Otimizacao)**: Regularizacao L1/L2 estudada la agora tem interpretacao bayesiana
- **1_4 (Regressao)**: Ridge e Lasso regression sao literalmente MAP com priors Normal e Laplace
- **1_2 (Inferencial)**: MLE frequentista e um caso especial de MAP sem prior

## 10. Exercicios Praticos

### Exercicio 1: Atualizacao Bayesiana Sequencial

Voce observa uma moeda lancada 20 vezes com 15 caras.
1. Use prior Beta(2, 2) e calcule o posterior
2. Calcule a media e variancia do posterior
3. Calcule o intervalo de credibilidade 95%
4. Compare com o MLE frequentista

In [ ]:
# TAREFA DO ALUNO: Exercicio 1 - Atualizacao Bayesiana
heads = 15
tails = 5
alpha_prior = 2
beta_prior = 2

# 1. Posterior
# alpha_post = None  # TAREFA DO ALUNO: alpha_prior + heads
# beta_post = None  # TAREFA DO ALUNO: beta_prior + tails

# 2. Media e variancia do posterior
# mean_post = None  # TAREFA DO ALUNO
# var_post = None  # TAREFA DO ALUNO

# 3. IC de credibilidade 95%
# ic_lower = None  # TAREFA DO ALUNO: beta.ppf(0.025, alpha_post, beta_post)
# ic_upper = None  # TAREFA DO ALUNO: beta.ppf(0.975, alpha_post, beta_post)

# 4. MLE
# mle = None  # TAREFA DO ALUNO: heads / (heads + tails)

# Imprima todos os resultados e compare

In [ ]:
# SOLUCAO - Exercicio 1
heads = 15
tails = 5
alpha_prior = 2
beta_prior = 2

print('=== EXERCICIO 1: ATUALIZACAO BAYESIANA ===')

# 1. Posterior
alpha_post = alpha_prior + heads
beta_post = beta_prior + tails
print(f'\nPrior: Beta({alpha_prior}, {beta_prior})')
print(f'Dados: {heads} caras, {tails} coroas')
print(f'Posterior: Beta({alpha_post}, {beta_post})')

# 2. Media e variancia
mean_post = alpha_post / (alpha_post + beta_post)
var_post = (alpha_post * beta_post) / ((alpha_post + beta_post)**2 * (alpha_post + beta_post + 1))
print(f'\nMedia posterior: {mean_post:.3f}')
print(f'Variancia posterior: {var_post:.4f}')
print(f'Desvio padrao posterior: {np.sqrt(var_post):.4f}')

# 3. IC de credibilidade
ic_lower = beta.ppf(0.025, alpha_post, beta_post)
ic_upper = beta.ppf(0.975, alpha_post, beta_post)
print(f'\nIC de credibilidade 95%: [{ic_lower:.3f}, {ic_upper:.3f}]')

# 4. MLE
mle = heads / (heads + tails)
print(f'\nComparacao:')
print(f'  MLE (frequentista): {mle:.3f}')
print(f'  Media posterior (bayesiano): {mean_post:.3f}')
print(f'  Diferenca: {abs(mle - mean_post):.3f} (prior puxa para 0.5)')

### Exercicio 2: Naive Bayes Manual

Crie um classificador Naive Bayes simples para o dataset abaixo:

| Weather | Temp | Play Tennis |
|---------|------|-------------|
| Sunny | Hot | No |
| Sunny | Hot | No |
| Overcast | Hot | Yes |
| Rainy | Mild | Yes |
| Rainy | Cool | Yes |
| Rainy | Cool | No |
| Overcast | Cool | Yes |
| Sunny | Mild | Yes |

Calcule manualmente:
1. Priors P(Yes) e P(No)
2. Likelihoods P(Weather|classe) e P(Temp|classe)
3. Classifique: Weather=Sunny, Temp=Cool

In [ ]:
# TAREFA DO ALUNO: Exercicio 2 - Naive Bayes Manual
import pandas as pd

data = pd.DataFrame({
    'Weather': ['Sunny', 'Sunny', 'Overcast', 'Rainy', 'Rainy', 'Rainy', 'Overcast', 'Sunny'],
    'Temp': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 'Mild'],
    'PlayTennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'Yes']
})

# 1. Priors
# prior_yes = None  # TAREFA DO ALUNO: contagem de Yes / total
# prior_no = None  # TAREFA DO ALUNO

# 2. Likelihoods para Weather=Sunny e Temp=Cool
# P(Sunny|Yes) = None  # TAREFA DO ALUNO
# P(Sunny|No) = None  # TAREFA DO ALUNO
# P(Cool|Yes) = None  # TAREFA DO ALUNO
# P(Cool|No) = None  # TAREFA DO ALUNO

# 3. Posterior (nao normalizado)
# score_yes = None  # TAREFA DO ALUNO: prior_yes * P(Sunny|Yes) * P(Cool|Yes)
# score_no = None  # TAREFA DO ALUNO: prior_no * P(Sunny|No) * P(Cool|No)

# Normalizar e imprimir resultado

In [ ]:
# SOLUCAO - Exercicio 2
data = pd.DataFrame({
    'Weather': ['Sunny', 'Sunny', 'Overcast', 'Rainy', 'Rainy', 'Rainy', 'Overcast', 'Sunny'],
    'Temp': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 'Mild'],
    'PlayTennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'Yes']
})

print('=== EXERCICIO 2: NAIVE BAYES MANUAL ===')
print(f'\nDataset:')
print(data)

# 1. Priors
play_counts = data['PlayTennis'].value_counts()
prior_yes = play_counts['Yes'] / len(data)
prior_no = play_counts['No'] / len(data)
print(f'\n1. Priors:')
print(f'   P(Yes) = {play_counts["Yes"]}/{len(data)} = {prior_yes:.3f}')
print(f'   P(No) = {play_counts["No"]}/{len(data)} = {prior_no:.3f}')

# 2. Likelihoods
n_yes = play_counts['Yes']
n_no = play_counts['No']

p_sunny_yes = len(data[(data['Weather']=='Sunny') & (data['PlayTennis']=='Yes')]) / n_yes
p_sunny_no = len(data[(data['Weather']=='Sunny') & (data['PlayTennis']=='No')]) / n_no
p_cool_yes = len(data[(data['Temp']=='Cool') & (data['PlayTennis']=='Yes')]) / n_yes
p_cool_no = len(data[(data['Temp']=='Cool') & (data['PlayTennis']=='No')]) / n_no

print(f'\n2. Likelihoods:')
print(f'   P(Sunny|Yes) = {p_sunny_yes:.3f}')
print(f'   P(Sunny|No) = {p_sunny_no:.3f}')
print(f'   P(Cool|Yes) = {p_cool_yes:.3f}')
print(f'   P(Cool|No) = {p_cool_no:.3f}')

# 3. Posterior
score_yes = prior_yes * p_sunny_yes * p_cool_yes
score_no = prior_no * p_sunny_no * p_cool_no
total = score_yes + score_no

print(f'\n3. Classificacao (Weather=Sunny, Temp=Cool):')
print(f'   Score(Yes) = {prior_yes:.3f} * {p_sunny_yes:.3f} * {p_cool_yes:.3f} = {score_yes:.4f}')
print(f'   Score(No) = {prior_no:.3f} * {p_sunny_no:.3f} * {p_cool_no:.3f} = {score_no:.4f}')
print(f'   P(Yes|dados) = {score_yes/total:.3f}')
print(f'   P(No|dados) = {score_no/total:.3f}')
print(f'   Predicao: {"YES" if score_yes > score_no else "NO"}')

### Exercicio 3: Sensibilidade ao Prior

Para o problema do teste medico (doenca com prevalencia p, teste com sensibilidade 95% e especificidade 95%):
1. Calcule P(doenca | teste+) para prevalencias: 0.1%, 1%, 5%, 10%, 50%
2. Plote a relacao entre prevalencia (prior) e posterior
3. Identifique a partir de qual prevalencia o teste se torna "util" (posterior > 50%)

In [ ]:
# TAREFA DO ALUNO: Exercicio 3 - Sensibilidade ao Prior
sensitivity = 0.95
specificity = 0.95

prevalences = [0.001, 0.01, 0.05, 0.10, 0.50]

# Para cada prevalencia, calcule P(doenca | teste+)
# posteriors = []
# for prev in prevalences:
#     # likelihood_pos_disease = sensitivity
#     # likelihood_pos_no_disease = 1 - specificity
#     # evidence = None  # TAREFA DO ALUNO
#     # posterior = None  # TAREFA DO ALUNO: Bayes
#     # posteriors.append(posterior)

# Encontre a prevalencia onde posterior > 0.5
# threshold_prev = None  # TAREFA DO ALUNO

# Plote prevalencia vs posterior

In [ ]:
# SOLUCAO - Exercicio 3
print('=== EXERCICIO 3: SENSIBILIDADE AO PRIOR ===')

sensitivity = 0.95
specificity = 0.95
prevalences = [0.001, 0.01, 0.05, 0.10, 0.50]

print(f'Sensibilidade: {sensitivity:.0%}, Especificidade: {specificity:.0%}')
print(f'\nPrevalencia -> P(doenca | teste+):')

posteriors = []
for prev in prevalences:
    lik_pos_disease = sensitivity
    lik_pos_no_disease = 1 - specificity
    evidence = lik_pos_disease * prev + lik_pos_no_disease * (1 - prev)
    posterior = (lik_pos_disease * prev) / evidence
    posteriors.append(posterior)
    print(f'  {prev:6.1%} -> {posterior:.3f} ({posterior:.1%})')

# Encontrar threshold
prev_range = np.linspace(0.001, 0.99, 1000)
post_range = [(sensitivity * p) / (sensitivity * p + (1-specificity) * (1-p)) for p in prev_range]
threshold_idx = np.argmin(np.abs(np.array(post_range) - 0.5))
threshold_prev = prev_range[threshold_idx]
print(f'\nPrevalencia onde posterior = 50%: ~{threshold_prev:.1%}')

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(prev_range, post_range, 'b-', linewidth=2, label='P(doenca | teste+)')
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='50% threshold')
ax.axvline(threshold_prev, color='green', linestyle='--', alpha=0.5, label=f'Prevalencia = {threshold_prev:.1%}')

for prev, post in zip(prevalences, posteriors):
    ax.plot(prev, post, 'ro', markersize=8)
    ax.annotate(f'{post:.1%}', (prev, post), textcoords="offset points", xytext=(10, 5))

ax.set_xlabel('Prevalencia (Prior)')
ax.set_ylabel('P(doenca | teste+) (Posterior)')
ax.set_title('Sensibilidade do Posterior ao Prior')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 0.5)
plt.tight_layout()
plt.show()

print(f'\nConclusao: Com teste 95% acurado, o resultado so e "confiavel"')
print(f'(posterior > 50%) quando a prevalencia e maior que ~{threshold_prev:.1%}')

### O que observar nos exercicios

- Exercicio 1: O posterior Beta(17,7) tem media 0.708, mais proxima de 0.75 (MLE) do que de 0.5 (prior), porque n=20 ja e bastante dado para prior fraco Beta(2,2)
- Exercicio 2: Naive Bayes manual mostra que a classificacao depende tanto dos priors (proporcoes de classe) quanto das likelihoods (probabilidades condicionais de features)
- Exercicio 3: A relacao prevalencia vs posterior e uma curva sigmoide - com teste 95% acurado, prevalencia precisa ser ~5% para resultado positivo ser confiavel

### O que concluir dos exercicios

- A forca do prior diminui conforme n cresce: com n=20, Beta(2,2) tem pouca influencia; com n=5, teria muita
- Naive Bayes manual e simples de calcular: basta contar frequencias condicionais e multiplicar
- A analise de sensibilidade ao prior e essencial: a mesma evidencia (teste positivo) leva a conclusoes opostas dependendo da prevalencia

## 11. Erros Comuns e Armadilhas

### Erro 1: Escolher prior arbitrariamente

**Errado**: "Vou usar prior Normal(0, 100) porque parece grande"
**Correto**: Justifique o prior com conhecimento do dominio ou use priors fracamente informativos. Faca analise de sensibilidade: mude o prior e veja se a conclusao muda.

### Erro 2: Confundir posterior com probabilidade verdadeira

**Errado**: "O posterior diz que P(theta=0.7) e alta, logo a moeda TEM 70% de chance de cara"
**Correto**: O posterior reflete SUA CRENCA dado seus dados e prior. Se mudar o prior, muda o posterior. E uma medida de incerteza, nao uma verdade absoluta.

### Erro 3: Usar prior muito forte sem justificativa

**Errado**: "Tenho certeza que a moeda e justa, vou usar Beta(1000, 1000)"
**Correto**: Um prior muito concentrado ignora os dados. Com Beta(1000,1000), precisaria de milhares de observacoes para mudar a crenca. Comece com priors fracos a menos que tenha FORTE justificativa.

### Erro 4: Achar que Bayesian resolve tudo automaticamente

**Errado**: "Nao preciso checar pressupostos ou validar porque sou bayesiano"
**Correto**: Modelos bayesianos tambem precisam de diagnosticos de convergencia (em MCMC), verificacao de pressupostos, e validacao preditiva.

### Erro 5: Ignorar a suposicao de independencia em Naive Bayes

**Errado**: "Naive Bayes assume independencia, logo nao funciona com features correlacionadas"
**Correto**: A suposicao e QUASE NUNCA verdadeira, mas o classificador funciona bem empiricamente. As probabilidades podem ser mal calibradas, mas as classificacoes sao frequentemente corretas.

### Erro 6: Nao usar prior conjugado quando disponivel

**Errado**: "Vou implementar MCMC para estimar uma proporcao binomial"
**Correto**: Para Beta-Binomial e Normal-Normal, existem solucoes ANALITICAS exatas. Use MCMC apenas quando necessario (modelos complexos sem conjugado).

### Erro 7: Confundir IC frequentista com IC bayesiano

**Errado**: "O IC 95% significa que ha 95% de chance do parametro estar nele"
**Correto**: Isso e a interpretacao do IC de CREDIBILIDADE (bayesiano). O IC de CONFIANCA (frequentista) tem interpretacao diferente: propriedade do procedimento, nao do parametro.

## 12. Resumo e Conexoes

### Hierarquia dos Conceitos

```
PRIOR: Crenca Inicial sobre o Parametro
    |
    | (Teorema de Bayes: incorpora dados)
    v
LIKELIHOOD: P(dados | parametro)
    |
    | (multiplicacao + normalizacao)
    v
POSTERIOR: Crenca Atualizada P(parametro | dados)
    |
    |---> MAP: ponto modal do posterior (estimativa pontual)
    |---> Media posterior: esperanca do posterior
    |---> Credible Interval: intervalo com probabilidade direta
    |---> Predicao: posterior preditivo para dados futuros
    |
    v
PRIORS CONJUGADOS (solucoes analiticas)
    |
    |---> Beta-Binomial: proporcoes (taxas de conversao, CTR)
    |---> Normal-Normal: medias (altura, peso, acuracia)
    |---> Atualizacao SEQUENCIAL: posterior de hoje = prior de amanha
    |
    v
APLICACOES EM ML
    |
    |---> Naive Bayes: classificacao via Bayes + independencia
    |---> Regularizacao: L2 = prior Normal, L1 = prior Laplace
    |---> Bayesian Optimization: prior sobre funcao objetivo
    |---> Online Learning: atualizacao sequencial de modelos
```

### Tabela de Conexoes

| Conceito deste notebook | Conecta com | Como |
|------------------------|-------------|------|
| Teorema de Bayes | 0_5 (Probabilidade) | Probabilidade condicional e a base matematica |
| Prior/Posterior | 0_6 (Distribuicoes) | Beta, Normal sao as distribuicoes mais usadas |
| IC de Credibilidade | 1_2 (Inferencial) | Alternativa bayesiana ao IC frequentista |
| Atualizacao sequencial | 0_7 (TCL) | Convergencia do posterior e analoga ao TCL |
| MAP = Regularizacao | 0_8 (Otimizacao) | L2=prior Normal, L1=prior Laplace |
| MAP = Regularizacao | 1_4 (Regressao) | Ridge = MAP Normal, Lasso = MAP Laplace |
| Naive Bayes | 1_1 (Descritiva) | Media e variancia por classe sao descritivas condicionais |
| A/B test bayesiano | 1_2 (Inferencial) | Resolve "peeking" do A/B test frequentista |
| Naive Bayes texto | 0_3 (Algebra Linear) | Bag of words e representacao vetorial |

### Checklist de Competencias

- [ ] Sei explicar a diferenca entre frequentista e bayesiano
- [ ] Sei aplicar o teorema de Bayes numericamente (diagnostico medico)
- [ ] Sei usar priors conjugados Beta-Binomial para atualizacao sequencial
- [ ] Sei implementar e interpretar Naive Bayes (Gaussian, Multinomial)
- [ ] Sei distinguir credible interval de confidence interval
- [ ] Sei explicar a conexao MAP = MLE + regularizacao
- [ ] Sei escolher entre prior forte e fraco baseado no contexto
- [ ] Sei fazer analise de sensibilidade ao prior

### Proximos Passos

No notebook **1_4 (Regressao Estatistica)**, voce vera como todos os conceitos de inferencia (frequentista e bayesiana) se combinam na analise de regressao. Coeficientes de regressao sao testados com testes t (1_2), intervalos de confianca sao calculados para predicoes, e regularizacao (Ridge/Lasso) e exatamente MAP bayesiano com priors Normal/Laplace.